In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!pip install -q -U \
unsloth \
transformers \
trl \
datasets \
accelerate \
peft \
bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 25.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 16.9 MB/s eta 0:00:00
   

In [4]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [8]:
dataset = load_dataset(
    "json",
    data_files="/kaggle/input/datasets/akra1234/government/government_chat_train.jsonl",
    split="train",
)

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:
EOS_TOKEN = tokenizer.eos_token

SYSTEM_PROMPT = (
    "তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। "
    "শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। "
    "যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না'।"
)

def formatting(example):
    instruction = example["instruction"].strip()
    input_text = example["input"].strip()
    answer = example["output"].strip()

    if input_text:
        user = instruction + "\n" + input_text
    else:
        user = instruction

    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{answer}<|im_end|>"
        + EOS_TOKEN
    )

    return {"text": text}

dataset = dataset.map(
    formatting,
    remove_columns=dataset.column_names,
)

Map:   0%|          | 0/1186 [00:00<?, ? examples/s]

In [10]:
print(dataset[0]["text"])

<|im_start|>system
তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না'।<|im_end|>
<|im_start|>user
NID আবেদন করতে কোথায় যেতে হবে?<|im_end|>
<|im_start|>assistant
আপনাকে https://services.nidw.gov.bd/nid-pub/citizen-home/apply ওয়েবসাইটে গিয়ে আবেদন করতে হবে এবং এপয়েন্টমেন্ট নিতে হবে।<|im_end|><|im_end|>


In [12]:
import torch
import torch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=True,          # Faster training
    args=SFTConfig(
        output_dir="./qwen-government",

        # Training
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # Optimizer
        learning_rate=2e-4,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,

        # Precision
        fp16=False,
        bf16=torch.cuda.is_bf16_supported(),

        # Logging
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",

        seed=3407,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1186 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [13]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,186 | Num Epochs = 3 | Total steps = 447
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.267876
20,0.641014
30,0.421363
40,0.344357
50,0.315797
60,0.318150
70,0.279136
80,0.262425
90,0.266937
100,0.254188


Unsloth: Restored added_tokens_decoder metadata in ./qwen-government/checkpoint-149/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-government/checkpoint-298/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-government/checkpoint-447/tokenizer_config.json.


TrainOutput(global_step=447, training_loss=0.20297298735420174, metrics={'train_runtime': 4325.285, 'train_samples_per_second': 0.823, 'train_steps_per_second': 0.103, 'total_flos': 5.321342454485299e+16, 'train_loss': 0.20297298735420174, 'epoch': 3.0})

In [14]:
model.save_pretrained("government_qwen_lora")
tokenizer.save_pretrained("government_qwen_lora")

Unsloth: Restored added_tokens_decoder metadata in government_qwen_lora/tokenizer_config.json.


('government_qwen_lora/tokenizer_config.json',
 'government_qwen_lora/chat_template.jinja',
 'government_qwen_lora/tokenizer.json')

In [15]:
import shutil

shutil.make_archive(
    "/kaggle/working/government_qwen_lora",
    "zip",
    "/kaggle/working/government_qwen_lora"
)

print("Created:", "/kaggle/working/government_qwen_lora.zip")

Created: /kaggle/working/government_qwen_lora.zip


In [16]:
!pip -q install bert-score rouge-score nltk rapidfuzz
import pandas as pd
import torch
import re

from rapidfuzz import fuzz
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import nltk
nltk.download("wordnet")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.6 MB/s eta 0:00:00a 0:00:01


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [18]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
LORA_MODEL = "/kaggle/working/government_qwen_lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    LORA_MODEL
)

model.eval()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
                (ba

In [20]:
df = pd.read_csv(
    "/kaggle/input/datasets/akra1234/government/merged_test_data.csv"
)

df.head()

,id,domain,topic,question_type,instruction,input,output,source_url,split,source
0,nid_003,nid,nid_number_structure,documents,NID আবেদন করতে কী কী ডকুমেন্ট লাগে?,NaN,"প্রিন্টেড আবেদনপত্র, পাসপোর্ট সাইজ ছবি, ১৭ ডিজ...",https://services.nidw.gov.bd/,test,NID
1,nid_006,nid,eligibility,procedure,NID কারা আবেদন করতে পারবে?,NaN,০১ অক্টোবর ২০১০ এর আগে জন্মগ্রহণকারী বাংলাদেশী...,https://services.nidw.gov.bd/,test,NID
2,nid_007,nid,new_voter_registration,procedure,যদি আগে ভোটার হয়ে থাকি তাহলে কি আবার আবেদন করত...,NaN,"না, আগে ভোটার হয়ে থাকলে নতুন নিবন্ধনের প্রয়োজন...",https://services.nidw.gov.bd/,test,NID
3,nid_014,nid,address_correction,documents,NID তথ্য হালনাগাদের জন্য কী কী প্রয়োজন?,NaN,তথ্য হালনাগাদের জন্য প্রয়োজনীয় প্রমাণপত্র যেমন...,https://services.nidw.gov.bd/,test,NID
4,nid_016,nid,nid_verification_status,procedure,NID রেজিস্ট্রেশন প্রক্রিয়া কী?,NaN,"রেজিস্ট্রেশন প্রক্রিয়ায় অনলাইন আবেদন, তথ্য যাচ...",https://services.nidw.gov.bd/,test,NID


In [21]:
SYSTEM = (
    "তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। "
    "শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। "
    "যদি জানা না থাকে বলবে 'আমি জানি না'।"
)

def generate_answer(question, context=""):

    if pd.isna(context):
        context = ""

    if len(context.strip()) > 0:
        user = question + "\n" + context
    else:
        user = question

    messages = [
        {
            "role":"system",
            "content":SYSTEM
        },
        {
            "role":"user",
            "content":user
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.0
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [22]:
predictions = []

for _, row in df.iterrows():

    pred = generate_answer(
        row["instruction"],
        row["input"]
    )

    predictions.append(pred)

df["prediction"] = predictions

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


AttributeError: 'Qwen2Attention' object has no attribute 'apply_qkv'

In [1]:
!pip install -q -U unsloth transformers trl peft accelerate bitsandbytes

In [2]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/government_qwen_lora",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("Model ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model ready


In [4]:
def generate_answer(instruction, input_text):

    if pd.isna(input_text):
        input_text = ""

    if input_text.strip():
        user_text = instruction + "\n" + input_text
    else:
        user_text = instruction


    messages = [
        {
            "role":"system",
            "content":
            "তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। "
            "শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। "
            "যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না'।"
        },
        {
            "role":"user",
            "content":user_text
        }
    ]


    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
            use_cache=True
        )


    output = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )


    return output.strip()

In [7]:
print(
    generate_answer(
        "NID আবেদন করতে কোথায় যেতে হবে?",
        ""
    )
)

NameError: name 'pd' is not defined

In [8]:
predictions=[]

for i,row in df.iterrows():

    pred=generate_answer(
        row["instruction"],
        row["input"]
    )

    predictions.append(pred)

    print(i, pred[:80])


df["prediction"]=predictions

NameError: name 'df' is not defined

In [ ]:
def normalize(text):

    text = str(text).lower()

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def exact_match(pred, ref):

    return normalize(pred) == normalize(ref)


def token_f1(pred, ref):

    pred_tokens = normalize(pred).split()

    ref_tokens = normalize(ref).split()

    common = set(pred_tokens) & set(ref_tokens)

    if len(common)==0:
        return 0

    precision = len(common)/len(pred_tokens)

    recall = len(common)/len(ref_tokens)

    return 2*precision*recall/(precision+recall)